# ADCS-ADL 21문항 EDA (수정판)

**이전 버전에서 수정한 점**: 이전엔 `adl_wide.csv`에 있는 `ADL01xx` 원컬럼 34개를 전부 "문항"으로 잡았는데, 그중 실제로 채점에 쓰이는 ADCS-ADL 문항은 21개뿐입니다. 나머지 13개(`ADL0108A/B/C`, `ADL0109A`, `ADL0119A/B/C`, `ADL0120A/B`, `ADL0124`, `ADL0125`)는 척도 문항이 아니거나 다른 하위질문이라 `b2_item_reduction_v6_communality.ipynb`에서도 `ITEMS`에서 제외되어 있었습니다. 그리고 `ADL0118A/B/C` 세 컬럼은 원래 **Q18 한 문항**이 세 개 하위질문으로 쪼개져 있던 것이라, 그 노트북과 동일하게 `__resolved` 세 값의 평균으로 합쳐서 Q18 하나로 씁니다.

이번 버전은 그 노트북의 `ITEMS` / `LAB` / `BADL` / `IADL` 정의를 그대로 가져와 문항명을 다시 뽑았습니다.

- **BADL(기본 일상생활, 6개)**: 먹기·걷기·화장실·목욕·몸단장·옷입기
- **IADL(도구적 일상생활, 15개)**: 옷고르기·전화·설거지·식사준비·집안일·빨래·가전·외출·쇼핑·지불·금전·혼자있기(Q18)·글쓰기·취미·가전사용
- **값 소스**: 관문(gate)이 있는 문항은 `__resolved`(관문 미통과 → 0 매입) 값을, 관문 구조가 없는 기본 5문항(Q1~Q5)은 원값을 그대로 사용합니다. (Q1~Q5는 `__gate`가 전부 `other`라 관문 로직 자체가 없습니다.)
- 척도에서 제외된 13개 원컬럼은 마지막에 **부록**으로 따로 요약만 붙여뒀습니다 (조용히 누락시키지 않기 위해).

Colab 사용 가정: 이 노트북과 `adl_wide.csv`를 같은 폴더에 두고 상대경로로 불러옵니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install koreanize-matplotlib -q

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 그래프 한글 깨짐 방지

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)

DATA_DIR = Path("/content/drive/MyDrive/2026 urp/preprocessed")
DATA_PATH = DATA_DIR / "adl_wide.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 1. 21문항 정의 (b2 FA 노트북과 동일)

`b2_item_reduction_v6_communality.ipynb`의 `ITEMS`/`LAB`/`BADL`/`IADL`을 그대로 가져왔습니다. 이 정의는 그 노트북에서도 "제가 추측/재구성한 부분"이라고 표시되어 있던 것이니, 혹시 정식 채점표(Table S4·S6·S7·S8) 기준과 다르면 이 셀만 고치면 아래 전체가 따라 바뀝니다.


In [ ]:
BADL = ["ADL0101", "ADL0102", "ADL0103", "ADL0104", "ADL0105", "ADL0106B"]
IADL = [
    "ADL0106A", "ADL0107A", "ADL0110A", "ADL0111A", "ADL0112A", "ADL0113A",
    "ADL0114A", "ADL0115A", "ADL0116A", "ADL0116B", "ADL0117A", "Q18",
    "ADL0121A", "ADL0122Q", "ADL0123L",
]
ITEMS = BADL + IADL  # 21문항
NI = len(ITEMS)

LAB = {
    "ADL0101": "Q1먹기", "ADL0102": "Q2걷기", "ADL0103": "Q3화장실", "ADL0104": "Q4목욕",
    "ADL0105": "Q5몸단장", "ADL0106B": "Q6b옷입기", "ADL0106A": "Q6a옷고르기", "ADL0107A": "Q7전화",
    "ADL0110A": "Q10설거지", "ADL0111A": "Q11식사준비", "ADL0112A": "Q12집안일", "ADL0113A": "Q13빨래",
    "ADL0114A": "Q14가전", "ADL0115A": "Q15외출", "ADL0116A": "Q16a쇼핑", "ADL0116B": "Q16b지불",
    "ADL0117A": "Q17금전", "Q18": "Q18혼자있기", "ADL0121A": "Q21글쓰기", "ADL0122Q": "Q22취미",
    "ADL0123L": "Q23가전사용",
}

# Q18: ADL0118A/B/C 세 하위질문을 __resolved 평균으로 합쳐서 하나의 문항으로 구성
Q18_SUBITEMS = ["ADL0118A__resolved", "ADL0118B__resolved", "ADL0118C__resolved"]
missing_sub = [c for c in Q18_SUBITEMS if c not in df.columns]
if missing_sub:
    print("⚠️ Q18 하위문항 중 없는 컬럼:", missing_sub)
else:
    df["Q18"] = df[Q18_SUBITEMS].mean(axis=1)

# 값 소스 결정: __resolved가 있으면 그걸, 없으면(Q1~Q5, Q18) 그대로
value_col = {}
for it in ITEMS:
    if it == "Q18":
        value_col[it] = "Q18"
    elif (it + "__resolved") in df.columns:
        value_col[it] = it + "__resolved"
    else:
        value_col[it] = it

missing_items = [it for it in ITEMS if value_col[it] not in df.columns]
if missing_items:
    print("⚠️ 데이터에 없는 문항(제외하고 진행):", missing_items)
    ITEMS = [it for it in ITEMS if it not in missing_items]
    BADL = [it for it in BADL if it in ITEMS]
    IADL = [it for it in IADL if it in ITEMS]
    NI = len(ITEMS)

print(f"최종 문항 수: {NI} (BADL {len(BADL)} + IADL {len(IADL)})")
pd.DataFrame({
    "코드": ITEMS,
    "라벨": [LAB[it] for it in ITEMS],
    "영역": ["BADL" if it in BADL else "IADL" for it in ITEMS],
    "사용컬럼": [value_col[it] for it in ITEMS],
})


## 2. 문항별 평균 / 분산 / 결측률

BADL(기본 일상생활)과 IADL(도구적 일상생활)을 구분해서 봅니다.


In [ ]:
item_summary = pd.DataFrame({
    "코드": ITEMS,
    "라벨": [LAB[it] for it in ITEMS],
    "영역": ["BADL" if it in BADL else "IADL" for it in ITEMS],
    "n_obs": [df[value_col[it]].count() for it in ITEMS],
    "n_missing": [df[value_col[it]].isna().sum() for it in ITEMS],
    "missing_rate": [round(df[value_col[it]].isna().mean(), 3) for it in ITEMS],
    "mean": [round(df[value_col[it]].mean(), 3) for it in ITEMS],
    "var": [round(df[value_col[it]].var(), 3) for it in ITEMS],
    "std": [round(df[value_col[it]].std(), 3) for it in ITEMS],
    "min": [df[value_col[it]].min() for it in ITEMS],
    "max": [df[value_col[it]].max() for it in ITEMS],
}).set_index("코드")

item_summary


In [ ]:
# 평균 ± SD 막대그래프 (BADL/IADL 색 구분)
colors = item_summary["영역"].map({"BADL": "#4C72B0", "IADL": "#DD8452"})
ordered = item_summary.sort_values(["영역", "mean"])

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(ordered["라벨"], ordered["mean"], yerr=ordered["std"], capsize=3,
       color=ordered["영역"].map({"BADL": "#4C72B0", "IADL": "#DD8452"}))
ax.set_xticks(range(len(ordered)))
ax.set_xticklabels(ordered["라벨"], rotation=60, ha="right")
ax.set_ylabel("평균 (오차막대 = 표준편차)")
ax.set_title("ADCS-ADL 21문항 평균 ± SD (파랑=BADL, 주황=IADL)")
plt.tight_layout()
plt.show()


In [ ]:
# 문항별 결측률 막대그래프
fig, ax = plt.subplots(figsize=(12, 5))
ordered_miss = item_summary.sort_values("missing_rate", ascending=False)
ax.bar(ordered_miss["라벨"], ordered_miss["missing_rate"] * 100,
       color=ordered_miss["영역"].map({"BADL": "#4C72B0", "IADL": "#DD8452"}))
ax.set_xticks(range(len(ordered_miss)))
ax.set_xticklabels(ordered_miss["라벨"], rotation=60, ha="right")
ax.set_ylabel("결측률 (%)")
ax.set_title("문항별 결측률")
plt.tight_layout()
plt.show()


## 3. 문항별 응답값 분포 (개별 막대그래프)

21개 문항을 그리드로 묶어서 각 문항이 어떤 값(0~4점 등급)들을 얼마나 갖는지 확인합니다.


In [ ]:
n_cols = 4
n_rows = int(np.ceil(NI / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

for i, it in enumerate(ITEMS):
    ax = axes[i]
    col = value_col[it]
    counts = df[col].value_counts(dropna=True).sort_index()
    domain_color = "#4C72B0" if it in BADL else "#DD8452"
    ax.bar(counts.index.astype(str), counts.values, color=domain_color)
    ax.set_title(f"{LAB[it]} ({it})", fontsize=9)
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7)

for j in range(NI, len(axes)):
    axes[j].axis("off")

fig.suptitle("문항별 응답값 분포 (파랑=BADL, 주황=IADL)", y=1.001)
plt.tight_layout()
plt.show()


## 4. BADL vs IADL 영역별 요약

문항을 하나로 묶어 영역(BADL/IADL) 단위로도 평균/분산을 봅니다.


In [ ]:
domain_summary = item_summary.groupby("영역")[["mean", "var", "missing_rate"]].mean().round(3)
domain_summary["n_items"] = item_summary.groupby("영역").size()
domain_summary


## 5. 문항 간 상관관계 (참고용)

이후 요인분석(FA)과 비교해볼 수 있게, 21문항 간 상관행렬 히트맵도 같이 봅니다.


In [ ]:
item_values = pd.DataFrame({LAB[it]: df[value_col[it]] for it in ITEMS})
corr = item_values.corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(NI)); ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(NI)); ax.set_yticklabels(corr.columns, fontsize=7)
fig.colorbar(im, ax=ax, label="상관계수")
ax.set_title("21문항 간 상관행렬")
plt.tight_layout()
plt.show()


## 6. 문항점수별 A1 단계(0~5) 분포

"이 문항에 N점을 준 사람들이 A1(2015판) 몇 단계에 있는가"를 봅니다. `adl_wide.csv`에 이미 `A1_2015_stage`가 같은 행에 있어서 별도 join 없이 바로 씁니다.

> 참고: 문항마다 점수 범위가 다릅니다 (예: Q1~Q5는 0~3점, Q6b/Q13/Q15/Q23은 0~4점, Q7은 0~5점, Q16b/Q18은 0/1). "0~4점"이라고 가정하지 않고 문항별 실제 관측 범위를 그대로 씁니다.

**행렬(교차표) 읽는 법**: 행=문항점수, 열=A1단계. 셀 값은 "그 문항점수를 받은 사람들 중 몇 %가 그 A1단계에 있는지"(행 기준 정규화)입니다. 즉 각 행의 합은 100%입니다.


In [ ]:
A1_LEVELS = list(range(0, 6))  # A1_2015_stage 범위 0~5

n_cols = 4
n_rows = int(np.ceil(NI / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3 * n_cols, 3.6 * n_rows))
axes = axes.flatten()

item_crosstabs = {}  # 나중에 재사용 가능하도록 저장

for i, it in enumerate(ITEMS):
    ax = axes[i]
    col = value_col[it]
    sub = df[[col, "A1_2015_stage"]].dropna().copy()
    sub[col] = sub[col].round().astype(int)
    sub["A1_2015_stage"] = sub["A1_2015_stage"].astype(int)

    ct = pd.crosstab(sub[col], sub["A1_2015_stage"])
    ct = ct.reindex(columns=A1_LEVELS, fill_value=0)
    row_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    item_crosstabs[it] = {"count": ct, "row_pct": row_pct}

    im = ax.imshow(row_pct.values, cmap="YlOrRd", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(len(A1_LEVELS)))
    ax.set_xticklabels(A1_LEVELS, fontsize=7)
    ax.set_yticks(range(len(row_pct)))
    ax.set_yticklabels(row_pct.index, fontsize=7)
    ax.set_title(LAB[it], fontsize=9)
    ax.set_xlabel("A1 단계", fontsize=7)
    ax.set_ylabel("문항점수", fontsize=7)

    for r in range(row_pct.shape[0]):
        for c in range(row_pct.shape[1]):
            v = row_pct.values[r, c]
            if v > 0:
                ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=6,
                        color="white" if v > 55 else "black")

for j in range(NI, len(axes)):
    axes[j].axis("off")

fig.suptitle("문항점수 → A1단계 분포 (행 기준 %, 문항점수를 준 사람 중 A1단계 구성)", y=1.001)
plt.tight_layout()
plt.show()


### 개별 문항 교차표를 표로도 보고 싶을 때

예시로 한 문항만 원(count)/비율(%) 표로 뽑아봅니다. `TARGET_ITEM`을 바꾸면 다른 문항도 볼 수 있습니다.


In [ ]:
TARGET_ITEM = "ADL0101"  # 원하는 문항 코드로 변경 (ITEMS 리스트 중 하나)

print(f"{LAB[TARGET_ITEM]} ({TARGET_ITEM}) — 원count")
display(item_crosstabs[TARGET_ITEM]["count"])

print(f"\n{LAB[TARGET_ITEM]} ({TARGET_ITEM}) — 행 기준 %")
display(item_crosstabs[TARGET_ITEM]["row_pct"].round(1))


## 7. 문항점수별 평균 A1 단계 (추세 확인용)

교차표보다 단순하게, 문항점수가 올라갈수록 평균 A1 단계가 어떻게 변하는지 선 그래프로 봅니다. 단조롭게 증가/감소하지 않는 문항이 있다면 그 문항의 변별력이 낮다는 신호로 볼 수 있습니다.


In [ ]:
n_cols = 4
n_rows = int(np.ceil(NI / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.6 * n_cols, 2.8 * n_rows))
axes = axes.flatten()

for i, it in enumerate(ITEMS):
    ax = axes[i]
    col = value_col[it]
    sub = df[[col, "A1_2015_stage"]].dropna().copy()
    sub[col] = sub[col].round().astype(int)
    grp = sub.groupby(col)["A1_2015_stage"].agg(["mean", "std", "count"])

    domain_color = "#4C72B0" if it in BADL else "#DD8452"
    ax.errorbar(grp.index, grp["mean"], yerr=grp["std"], marker="o", capsize=3, color=domain_color)
    ax.set_title(LAB[it], fontsize=9)
    ax.set_xlabel("문항점수", fontsize=7)
    ax.set_ylabel("평균 A1단계", fontsize=7)
    ax.set_ylim(-0.5, 5.5)
    ax.tick_params(labelsize=7)

for j in range(NI, len(axes)):
    axes[j].axis("off")

fig.suptitle("문항점수별 평균 A1단계 (오차막대=표준편차, 파랑=BADL, 주황=IADL)", y=1.001)
plt.tight_layout()
plt.show()


## 8. 문항-A1단계 상관관계 요약 (변별력 순위)

문항점수와 A1단계 간 Spearman 순위상관을 계산해 정렬합니다. 절댓값이 클수록 그 문항 하나만으로도 A1단계를 잘 구분한다는 뜻입니다 (요인분석의 공통성(h²)과는 다른, 더 단순한 지표입니다).


In [ ]:
from scipy.stats import spearmanr

rows = []
for it in ITEMS:
    col = value_col[it]
    sub = df[[col, "A1_2015_stage"]].dropna()
    rho, pval = spearmanr(sub[col], sub["A1_2015_stage"])
    rows.append({
        "코드": it, "라벨": LAB[it], "영역": "BADL" if it in BADL else "IADL",
        "spearman_rho": round(rho, 3), "p_value": pval, "n": len(sub),
    })

corr_summary = pd.DataFrame(rows).sort_values("spearman_rho")
corr_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
colors = corr_summary["영역"].map({"BADL": "#4C72B0", "IADL": "#DD8452"})
ax.barh(corr_summary["라벨"], corr_summary["spearman_rho"], color=colors)
ax.set_xlabel("Spearman ρ (문항점수 vs A1단계)")
ax.set_title("문항별 A1단계와의 상관관계 (파랑=BADL, 주황=IADL)")
ax.axvline(0, color="gray", linewidth=0.8)
plt.tight_layout()
plt.show()


## 부록. 척도에서 제외된 13개 원컬럼 (참고용)

`ADL0108A/B/C`, `ADL0109A`, `ADL0119A/B/C`, `ADL0120A/B`, `ADL0124`, `ADL0125` — 21문항 정의(`ITEMS`)에는 포함되지 않은 원컬럼들입니다. 척도 점수 계산에는 안 쓰지만, 데이터에 뭐가 있는지 조용히 누락시키지 않으려고 별도로만 요약해둡니다. (`ADL0124`는 값 종류가 77개로 다른 문항과 성격이 달라 보이니, 이 문항이 무엇을 묻는지는 코드북에서 별도 확인이 필요합니다.)


In [ ]:
excluded_raw = ["ADL0108A", "ADL0108B", "ADL0108C", "ADL0109A",
                "ADL0119A", "ADL0119B", "ADL0119C", "ADL0120A", "ADL0120B",
                "ADL0124", "ADL0125"]
excluded_raw = [c for c in excluded_raw if c in df.columns]

excl_summary = pd.DataFrame({
    "n_obs": df[excluded_raw].count(),
    "n_missing": df[excluded_raw].isna().sum(),
    "missing_rate": df[excluded_raw].isna().mean().round(3),
    "nunique": df[excluded_raw].nunique(),
    "mean": df[excluded_raw].mean(numeric_only=True),
    "std": df[excluded_raw].std(numeric_only=True),
    "min": df[excluded_raw].min(numeric_only=True),
    "max": df[excluded_raw].max(numeric_only=True),
})
excl_summary


## 6. 요약 저장


In [ ]:
item_summary.to_csv("adl_21item_summary.csv", encoding="utf-8-sig")
excl_summary.to_csv("adl_excluded_columns_summary.csv", encoding="utf-8-sig")
corr_summary.to_csv("adl_item_vs_A1_correlation.csv", index=False, encoding="utf-8-sig")
print("저장 완료: adl_21item_summary.csv, adl_excluded_columns_summary.csv, adl_item_vs_A1_correlation.csv")


# 개인내/개인간 변동성 분석

In [ ]:
score_cols = [value_col[it] for it in ITEMS]
MAXTOT = sum(df[c].max() for c in score_cols)

df["adl_total"] = df[score_cols].sum(axis=1, skipna=False)

print(f"문항 최대합(이론적 상한): {MAXTOT}")
print(f"완전관측 total row 수: {df['adl_total'].notna().sum()} / {len(df)}")
df[["USUBJID", "VISITNUM", "visit_label", "adl_total", "A1_2015_stage"]].head()

## 7. A1단계별 천장·바닥효과 먼저 확인

A1별 변동성을 비교하기 전에, 각 A1단계에서 총점이 상한/하한에 몰려있는지부터 봅니다. 몰려있으면 "변동성이 작다"는 결과가 진짜 안정적이어서가 아니라 점수 자체가 더 움직일 여지가 없어서(측정 한계)일 수 있습니다. baseline(VISITNUM=2.0) 기준으로 봅니다.

In [ ]:
CEIL_THRESH = 0.9
FLOOR_THRESH = 0.1

base = df[df["VISITNUM"] == 2.0].dropna(subset=["adl_total", "A1_2015_stage"]).copy()
base["near_ceiling"] = base["adl_total"] >= MAXTOT * CEIL_THRESH
base["near_floor"] = base["adl_total"] <= MAXTOT * FLOOR_THRESH

ceiling_floor_summary = base.groupby("A1_2015_stage").agg(
    n=("adl_total", "size"),
    mean_total=("adl_total", "mean"),
    sd_total=("adl_total", "std"),
    pct_near_ceiling=("near_ceiling", "mean"),
    pct_near_floor=("near_floor", "mean"),
).round(3)
ceiling_floor_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(ceiling_floor_summary.index.astype(str), ceiling_floor_summary["pct_near_ceiling"] * 100, color="#C44E52")
axes[0].set_title(f"A1단계별 천장효과 (총점≥{int(MAXTOT*CEIL_THRESH)}점 비율)")
axes[0].set_xlabel("A1 단계 (baseline)"); axes[0].set_ylabel("%")

axes[1].bar(ceiling_floor_summary.index.astype(str), ceiling_floor_summary["pct_near_floor"] * 100, color="#4C72B0")
axes[1].set_title(f"A1단계별 바닥효과 (총점≤{int(MAXTOT*FLOOR_THRESH)}점 비율)")
axes[1].set_xlabel("A1 단계 (baseline)"); axes[1].set_ylabel("%")

plt.tight_layout()
plt.show()

## 8. 개인별 방문 횟수 확인 및 필터링

개인내 변동성은 최소 2번 이상 방문이 있어야 계산되고, 변화율(기울기)도 시점이 적으면 신뢰도가 떨어집니다. 여기선 방문 3회 이상인 사람만 포함합니다(원하면 `MIN_VISITS` 값을 바꾸세요).

In [ ]:
visits_per_person = df.groupby("USUBJID")["VISITNUM"].nunique()
print(visits_per_person.value_counts().sort_index())

MIN_VISITS = 3
valid_ids = visits_per_person[visits_per_person >= MIN_VISITS].index
print(f"방문 {MIN_VISITS}회 이상인 사람: {len(valid_ids)} / {len(visits_per_person)}")

## 9. 그룹 라벨 — baseline A1단계

변동성 분석 기간(baseline~week24) 동안 A1단계 자체가 변하므로, 그룹을 나눌 기준은 **baseline 시점의 A1단계**로 고정합니다.

In [ ]:
baseline_a1 = df[df["VISITNUM"] == 2.0][["USUBJID", "A1_2015_stage"]].dropna()
baseline_a1 = baseline_a1.rename(columns={"A1_2015_stage": "baseline_A1"})
baseline_a1 = baseline_a1.drop_duplicates(subset="USUBJID")
baseline_a1.head()

## 10. 개인별 지표 계산

- `person_mean_total`: 개인내 총점 평균
- `person_sd_total`: 개인내 총점 변동성 (방문 간 표준편차)
- `person_mad_total`: 개인내 총점 평균편차 (평균으로부터의 절대편차 평균)
- `person_change_rate`: 개인내 총점 변화율 (첫 방문→마지막 방문, 방문 1회당 변화량)

In [ ]:
long = df[df["USUBJID"].isin(valid_ids)][["USUBJID", "VISITNUM", "adl_total"]].dropna(subset=["adl_total"]).copy()

def person_stats(g):
    g = g.sort_values("VISITNUM")
    n = len(g)
    mean_total = g["adl_total"].mean()
    sd_total = g["adl_total"].std() if n >= 2 else np.nan
    mad_total = (g["adl_total"] - mean_total).abs().mean()
    if n >= 2:
        first, last = g["adl_total"].iloc[0], g["adl_total"].iloc[-1]
        visit_span = g["VISITNUM"].iloc[-1] - g["VISITNUM"].iloc[0]
        change_rate = (last - first) / visit_span if visit_span > 0 else np.nan
    else:
        change_rate = np.nan
    return pd.Series({"n_visits": n, "person_mean_total": mean_total,
                       "person_sd_total": sd_total, "person_mad_total": mad_total,
                       "person_change_rate": change_rate})

person_metrics = long.groupby("USUBJID").apply(person_stats).reset_index()
person_metrics = person_metrics.merge(baseline_a1, on="USUBJID", how="left")

print(person_metrics.dropna(subset=["baseline_A1"]).groupby("baseline_A1").size())
person_metrics.head()

## 11. A1단계별 개인내 지표 비교

10번에서 본 천장/바닥효과를 염두에 두고 해석하세요 — 특히 A1=0,1은 천장효과가 커서 변동성이 작게 나오는 게 자연스럽습니다.

In [ ]:
plot_df = person_metrics.dropna(subset=["baseline_A1"])
metrics_to_plot = [
    ("person_mean_total", "개인내 총점 평균"),
    ("person_sd_total", "개인내 총점 변동성 (SD)"),
    ("person_mad_total", "개인내 총점 평균편차 (MAD)"),
    ("person_change_rate", "개인내 총점 변화율 (점/방문)"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()
levels = sorted(plot_df["baseline_A1"].unique())

for i, (col, title) in enumerate(metrics_to_plot):
    ax = axes[i]
    data = [plot_df.loc[plot_df["baseline_A1"] == lv, col].dropna() for lv in levels]
    ax.boxplot(data, tick_labels=[str(int(lv)) for lv in levels], showmeans=True)
    ax.set_title(title)
    ax.set_xlabel("baseline A1 단계")

plt.tight_layout()
plt.show()

## 12. A1단계별 개인간 변동성

각 A1그룹 안에서 사람들의 `person_mean_total`이 서로 얼마나 다른지(개인간 변동성)를 SD와 변동계수(CV = SD/평균)로 봅니다. CV를 같이 보는 이유: 그룹마다 평균 자체가 다르니(A1=0은 63점대, A1=5는 25점대) SD만 비교하면 절대적 스케일 차이에 오도될 수 있어서입니다.

In [ ]:
between_person = plot_df.groupby("baseline_A1")["person_mean_total"].agg(
    n="size", 개인간_평균="mean", 개인간_SD="std", 개인간_CV=lambda x: x.std()/x.mean()
).round(3)
between_person